
# 1. Train notebook — ESIM+ for NLI

This notebook does **training only**.

It uses the development split during training for:
- early stopping
- threshold search
- checkpoint selection

But it does **not** run the final standalone development-set evaluation report.
That is handled in the separate evaluation notebook.

## Saved output

This notebook saves **one file only**:
- `MODEL_BUNDLE_PATH` — a single PyTorch bundle containing:
  - model weights
  - vocabulary
  - model configuration
  - best threshold
  - best dev metrics from training
  - training history

That single bundle is all the evaluation notebook and demo notebook need.


In [10]:
!pip -q install torch pandas numpy scikit-learn matplotlib tqdm gensim

In [11]:

import os
import re
import json
import time
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_curve,
)



In [12]:
"""
ESIM-style NLI model

Core architecture:
- ESIM: Chen et al. (2017), "Enhanced LSTM for Natural Language Inference"
  https://aclanthology.org/P17-1152/

Extensions:
- learned gated pooling over the composed sequence
- sentence-level interaction features [u, v, |u-v|, u*v]
"""

from __future__ import annotations

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


def masked_softmax(scores: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Softmax that ignores padded positions."""
    scores = scores.masked_fill(~mask, -1e9)
    return torch.softmax(scores, dim=-1)


class GatedPooling(nn.Module):
    """
    Lightweight learned pooling.

    Plain ESIM uses avg/max pooling after the composition BiLSTM.
    Here we keep avg/max information, but also learn a token importance gate.
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.gate = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, dim], mask: [batch, seq_len]
        mask_f = mask.unsqueeze(-1).float()

        # Average pooling over valid tokens.
        avg_pool = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-6)

        # Max pooling over valid tokens.
        max_pool = x.masked_fill(~mask.unsqueeze(-1), -1e9).max(dim=1).values

        # Learned gated pooling.
        gate_scores = self.gate(x).squeeze(-1)                        # [batch, seq_len]
        gate_scores = gate_scores.masked_fill(~mask, -1e9)
        gate_weights = torch.softmax(gate_scores, dim=1).unsqueeze(-1)
        gated_pool = (x * gate_weights).sum(dim=1)

        return torch.cat([avg_pool, max_pool, gated_pool], dim=-1)


class ESIMPlus(nn.Module):
    """
    ESIM-style NLI encoder/classifier.

    Paper feature map
    -----------------
    Base ESIM features from Chen et al. (2017):
    - BiLSTM encoding
    - soft alignment between premise and hypothesis
    - local inference matching [x, y, x-y, x*y]
    - second BiLSTM composition layer

    Extensions:
    - learned gated pooling
    - sentence-level interaction features [u, v, |u-v|, u*v]

    Local inference vs sentence-level interaction
    ---------------------------------------------
    Local inference is token-level:
        compare each token to its aligned token summary in the other sentence.
    Sentence-level interaction is sequence-level:
        compare the final pooled premise vector u and pooled hypothesis vector v.
    """
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        padding_idx: int = 0,
        embedding_matrix: np.ndarray | None = None,
        dropout: float = 0.3,
        train_embeddings: bool = True,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = train_embeddings

        self.embedding_dropout = nn.Dropout(dropout)

        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        # ESIM local inference vectors have width 8h:
        # [x, y, x-y, x*y] and each x/y is 2h from the BiLSTM.
        self.projection = nn.Sequential(
            nn.Linear(hidden_size * 8, hidden_size),
            nn.ReLU(),
        )

        self.composition = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        composed_dim = hidden_size * 2
        pooled_dim = composed_dim * 3  # avg + max + gated

        self.pooler = GatedPooling(composed_dim)

        # Sentence-level interaction features:
        # u, v, |u-v|, u*v
        classifier_input_dim = pooled_dim * 4
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(classifier_input_dim, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 1),
        )

    def _run_bilstm(self, x: torch.Tensor, lengths: torch.Tensor, lstm: nn.LSTM):
        lstm.flatten_parameters()
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        return output

    def _apply_attention(
        self,
        a: torch.Tensor,
        a_mask: torch.Tensor,
        b: torch.Tensor,
        b_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Token-to-token similarity matrix.
        similarity = torch.matmul(a, b.transpose(1, 2))

        b_mask_expanded = b_mask.unsqueeze(1).expand(-1, a.size(1), -1)
        attn_a = masked_softmax(similarity, b_mask_expanded)
        attended_a = torch.matmul(attn_a, b)

        a_mask_expanded = a_mask.unsqueeze(1).expand(-1, b.size(1), -1)
        attn_b = masked_softmax(similarity.transpose(1, 2), a_mask_expanded)
        attended_b = torch.matmul(attn_b, a)

        return attended_a, attended_b

    def forward(
        self,
        premise_ids: torch.Tensor,
        premise_lengths: torch.Tensor,
        hypothesis_ids: torch.Tensor,
        hypothesis_lengths: torch.Tensor,
    ) -> torch.Tensor:
        premise_mask = premise_ids.ne(0)
        hypothesis_mask = hypothesis_ids.ne(0)

        premise_embed = self.embedding_dropout(self.embedding(premise_ids))
        hypothesis_embed = self.embedding_dropout(self.embedding(hypothesis_ids))

        premise_encoded = self._run_bilstm(premise_embed, premise_lengths, self.encoder)
        hypothesis_encoded = self._run_bilstm(hypothesis_embed, hypothesis_lengths, self.encoder)

        premise_aligned, hypothesis_aligned = self._apply_attention(
            premise_encoded, premise_mask, hypothesis_encoded, hypothesis_mask
        )

        # Local inference matching from ESIM.
        premise_enhanced = torch.cat(
            [
                premise_encoded,
                premise_aligned,
                premise_encoded - premise_aligned,
                premise_encoded * premise_aligned,
            ],
            dim=-1,
        )
        hypothesis_enhanced = torch.cat(
            [
                hypothesis_encoded,
                hypothesis_aligned,
                hypothesis_encoded - hypothesis_aligned,
                hypothesis_encoded * hypothesis_aligned,
            ],
            dim=-1,
        )

        premise_projected = self.projection(premise_enhanced)
        hypothesis_projected = self.projection(hypothesis_enhanced)

        premise_composed = self._run_bilstm(premise_projected, premise_lengths, self.composition)
        hypothesis_composed = self._run_bilstm(hypothesis_projected, hypothesis_lengths, self.composition)

        premise_vector = self.pooler(premise_composed, premise_mask)
        hypothesis_vector = self.pooler(hypothesis_composed, hypothesis_mask)

        # Sentence-level interaction features.
        pair_vector = torch.cat(
            [
                premise_vector,
                hypothesis_vector,
                torch.abs(premise_vector - hypothesis_vector),
                premise_vector * hypothesis_vector,
            ],
            dim=-1,
        )

        logits = self.classifier(pair_vector).squeeze(-1)
        return logits


In [13]:

# ---------------------------
# User configuration
# ---------------------------
TRAIN_PATH = "train.csv"
DEV_PATH = "dev.csv"
MODEL_BUNDLE_PATH = "nli_esim_plus_bundle.pt"

MODEL_ID = "nli-esim-plus-category-b"
DEVELOPERS = "Mateusz Wojcieszyk"
LANGUAGE = "English"
TRACK_NAME = "Natural Language Inference (NLI)"
NOTEBOOK_VERSION = "train-v1"

SEED = 42

LOWERCASE = True
MAX_LEN = 128
MIN_FREQ = 2
MAX_VOCAB_SIZE = 50000

USE_PRETRAINED_EMBEDDINGS = True
EMBEDDING_BACKEND = "gensim"   # "gensim", "local_txt", or "random"
GENSIM_MODEL_NAME = "glove-wiki-gigaword-100"
LOCAL_EMBEDDING_PATH = None
EMBEDDING_DIM = 100
TRAIN_EMBEDDINGS = True

HIDDEN_SIZE = 192
DROPOUT = 0.3

BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 12
PATIENCE = 4
GRAD_CLIP = 5.0

# Optional regularisation
USE_RDROP = True
RDROP_ALPHA = 0.5

USE_SWA = True
SWA_START_EPOCH = 8
SWA_LR = 1e-4

# Optional shortlist search.
# Leave this False for a single clean final training run.
RUN_SHORTLIST_SEARCH = False
SEARCH_EPOCHS = 8
SEARCH_PATIENCE = 3
SEARCH_SHORTLIST = [
    {
        "hidden_size": 192,
        "dropout": 0.3,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "max_len": 128,
        "train_embeddings": True,
    },
    {
        "hidden_size": 256,
        "dropout": 0.3,
        "learning_rate": 3e-4,
        "weight_decay": 1e-5,
        "batch_size": 64,
        "max_len": 128,
        "train_embeddings": True,
    },
]

out_dir = os.path.dirname(MODEL_BUNDLE_PATH)
if out_dir:
    os.makedirs(out_dir, exist_ok=True)


In [14]:
# Utility functions for reproducibility and preprocessing.
# These functions are shared by training, evaluation, and the demo workflow so
# the exact same tokenisation and vocabulary logic is used everywhere.

def set_seed(seed: int = 42):
    """Set random seeds for Python, NumPy and PyTorch.
    This makes experiments easier to reproduce.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

TOKEN_RE = re.compile(r"\w+|[^\w\s]")

def tokenize(text: str):
    """Tokenise text with a lightweight regex tokenizer.
    Punctuation is kept as separate tokens because that can help NLI.
    """
    text = "" if pd.isna(text) else str(text)
    if LOWERCASE:
        text = text.lower()
    return TOKEN_RE.findall(text)

def build_vocab(train_df: pd.DataFrame, min_freq: int = 2, max_size: int = 50000):
    """Build a vocabulary from the training split only.
    Using train only avoids leaking information from the dev set.
    """
    counter = Counter()
    for text in pd.concat([train_df["premise"], train_df["hypothesis"]], axis=0).tolist():
        counter.update(tokenize(text))

    vocab = {"<pad>": 0, "<unk>": 1}
    for token, freq in counter.most_common():
        if freq < min_freq:
            continue
        if len(vocab) >= max_size:
            break
        vocab[token] = len(vocab)
    return vocab, counter

def encode_text(text: str, vocab: dict, max_len: int):
    """Convert one text string into a tensor of token ids.
    Sequences are truncated to max_len to control memory and runtime.
    """
    token_ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokenize(text)[:max_len]]
    if not token_ids:
        token_ids = [vocab["<unk>"]]
    return torch.tensor(token_ids, dtype=torch.long)

def describe_lengths(df: pd.DataFrame):
    """Return simple token-length statistics for a dataframe."""
    lengths = [len(tokenize(x)) for x in pd.concat([df["premise"], df["hypothesis"]], axis=0).tolist()]
    return {
        "count": len(lengths),
        "mean": float(np.mean(lengths)),
        "p90": int(np.percentile(lengths, 90)),
        "p95": int(np.percentile(lengths, 95)),
        "max": int(np.max(lengths)),
    }

def load_local_txt_embeddings(txt_path: str, vocab: dict):
    """Load pretrained embeddings from a local .txt file and align them with the vocabulary."""
    if txt_path is None or not os.path.exists(txt_path):
        raise FileNotFoundError("Provide a valid LOCAL_EMBEDDING_PATH or switch backend.")

    embedding_index = {}
    embedding_dim = None
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if len(parts) <= 2:
                continue
            word = parts[0]
            vec = np.asarray(parts[1:], dtype=np.float32)
            if embedding_dim is None:
                embedding_dim = len(vec)
            embedding_index[word] = vec

    matrix = np.random.normal(0.0, 0.05, size=(len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab["<pad>"]] = 0.0

    found = 0
    for token, idx in vocab.items():
        vec = embedding_index.get(token)
        if vec is not None:
            matrix[idx] = vec
            found += 1

    return matrix, embedding_dim, found / max(1, len(vocab))

def load_gensim_embeddings(model_name: str, vocab: dict):
    """Load pretrained embeddings via gensim and align them with the vocabulary."""
    import gensim.downloader as api

    print(f"Loading pretrained vectors: {model_name}")
    vectors = api.load(model_name)
    embedding_dim = int(vectors.vector_size)

    matrix = np.random.normal(0.0, 0.05, size=(len(vocab), embedding_dim)).astype(np.float32)
    matrix[vocab["<pad>"]] = 0.0

    found = 0
    for token, idx in vocab.items():
        if token in vectors:
            matrix[idx] = vectors[token]
            found += 1

    return matrix, embedding_dim, found / max(1, len(vocab))

def build_embedding_matrix(vocab: dict):
    """Create the matrix used to initialise nn.Embedding.
    Missing tokens are randomly initialised.
    """
    if not USE_PRETRAINED_EMBEDDINGS or EMBEDDING_BACKEND == "random":
        matrix = np.random.normal(0.0, 0.05, size=(len(vocab), EMBEDDING_DIM)).astype(np.float32)
        matrix[vocab["<pad>"]] = 0.0
        return matrix, EMBEDDING_DIM, 0.0, "random"

    if EMBEDDING_BACKEND == "gensim":
        matrix, dim, coverage = load_gensim_embeddings(GENSIM_MODEL_NAME, vocab)
        return matrix, dim, coverage, f"gensim::{GENSIM_MODEL_NAME}"

    if EMBEDDING_BACKEND == "local_txt":
        matrix, dim, coverage = load_local_txt_embeddings(LOCAL_EMBEDDING_PATH, vocab)
        return matrix, dim, coverage, f"local_txt::{LOCAL_EMBEDDING_PATH}"

    raise ValueError("EMBEDDING_BACKEND must be one of: gensim, local_txt, random")


Using device: cuda


In [15]:
# Dataset and batch collation.
# The Dataset turns each CSV row into token-id tensors.
# The collate function pads each batch to the longest sequence in the batch and
# returns the true lengths needed by the BiLSTM encoder.

class NLIDataset(Dataset):
    """PyTorch dataset for premise-hypothesis pairs."""
    def __init__(self, dataframe: pd.DataFrame, vocab: dict, max_len: int = 128, with_labels: bool = True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.vocab = vocab
        self.max_len = max_len
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        premise_ids = encode_text(row["premise"], self.vocab, self.max_len)
        hypothesis_ids = encode_text(row["hypothesis"], self.vocab, self.max_len)

        if self.with_labels:
            label = torch.tensor(float(row["label"]), dtype=torch.float32)
            return premise_ids, hypothesis_ids, label

        return premise_ids, hypothesis_ids

def nli_collate(batch):
    """Pad a batch of variable-length sequences and return true lengths."""
    if len(batch[0]) == 3:
        premises, hypotheses, labels = zip(*batch)
    else:
        premises, hypotheses = zip(*batch)
        labels = None

    premise_lengths = torch.tensor([len(x) for x in premises], dtype=torch.long)
    hypothesis_lengths = torch.tensor([len(x) for x in hypotheses], dtype=torch.long)

    premises = pad_sequence(premises, batch_first=True, padding_value=0)
    hypotheses = pad_sequence(hypotheses, batch_first=True, padding_value=0)

    if labels is None:
        return premises, premise_lengths, hypotheses, hypothesis_lengths

    labels = torch.stack(labels)
    return premises, premise_lengths, hypotheses, hypothesis_lengths, labels

def build_dataloaders(train_df, dev_df, vocab, max_len, batch_size):
    train_dataset = NLIDataset(train_df, vocab=vocab, max_len=max_len, with_labels=True)
    dev_dataset = NLIDataset(dev_df, vocab=vocab, max_len=max_len, with_labels=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=nli_collate)
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, collate_fn=nli_collate)
    return train_loader, dev_loader


In [16]:
# Training regularisation and evaluation helpers.
# - R-Drop is optional and adds a consistency term between two dropout-enabled
#   forward passes.
# - Metric helpers keep training and evaluation logic consistent.

def bernoulli_symmetric_kl_from_logits(logits_a: torch.Tensor, logits_b: torch.Tensor) -> torch.Tensor:
    """Symmetric KL divergence used by R-Drop for binary logits."""
    p = torch.sigmoid(logits_a).clamp(1e-6, 1 - 1e-6)
    q = torch.sigmoid(logits_b).clamp(1e-6, 1 - 1e-6)

    kl_pq = p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))
    kl_qp = q * torch.log(q / p) + (1 - q) * torch.log((1 - q) / (1 - p))
    return 0.5 * (kl_pq + kl_qp).mean()

def compute_metrics_from_probs(y_true, y_prob, threshold: float = 0.5):
    """Convert probabilities into hard predictions and compute dev metrics."""
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=np.float32)
    y_pred = (y_prob >= threshold).astype(int)

    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1_macro),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }

    try:
        metrics["roc_auc"] = float(roc_auc_score(y_true, y_prob))
    except Exception:
        metrics["roc_auc"] = None

    return metrics, y_pred

def evaluate_model(model, dataloader, loss_fn, device, threshold: float = 0.5):
    """Run the model on a dataloader without gradient updates."""
    model.eval()
    all_probs = []
    all_labels = []
    total_loss = 0.0

    with torch.no_grad():
        for premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths, labels in dataloader:
            premise_ids = premise_ids.to(device)
            premise_lengths = premise_lengths.to(device)
            hypothesis_ids = hypothesis_ids.to(device)
            hypothesis_lengths = hypothesis_lengths.to(device)
            labels = labels.to(device)

            logits = model(premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths)
            loss = loss_fn(logits, labels)
            probs = torch.sigmoid(logits)

            total_loss += loss.item()
            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    avg_loss = total_loss / max(1, len(dataloader))
    metrics, preds = compute_metrics_from_probs(all_labels, all_probs, threshold=threshold)
    metrics["loss"] = float(avg_loss)
    return metrics, np.array(all_labels), np.array(all_probs), np.array(preds)

def find_best_threshold(y_true, y_prob, threshold_min: float = 0.30, threshold_max: float = 0.70, threshold_step: float = 0.02):
    """Search a small threshold range and keep the best macro-F1 value."""
    best_threshold = 0.5
    best_metrics, _ = compute_metrics_from_probs(y_true, y_prob, threshold=0.5)
    best_score = best_metrics["f1_macro"]

    thresholds = np.arange(threshold_min, threshold_max + 1e-9, threshold_step)
    sweep = []
    for threshold in thresholds:
        threshold = float(round(float(threshold), 2))
        metrics, _ = compute_metrics_from_probs(y_true, y_prob, threshold=threshold)
        sweep.append({"threshold": threshold, "f1_macro": metrics["f1_macro"], "accuracy": metrics["accuracy"]})
        if metrics["f1_macro"] > best_score:
            best_score = metrics["f1_macro"]
            best_threshold = threshold
            best_metrics = metrics

    return best_threshold, best_metrics, pd.DataFrame(sweep)

def build_model(vocab_size, embedding_dim, hidden_size, padding_idx, embedding_matrix, dropout, train_embeddings):
    return ESIMPlus(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        hidden_size=hidden_size,
        padding_idx=padding_idx,
        embedding_matrix=embedding_matrix,
        dropout=dropout,
        train_embeddings=train_embeddings,
    ).to(device)


In [17]:

# Main training loop.
# This function supports:
# - ordinary supervised training
# - optional R-Drop regularisation
# - optional SWA-style averaging near the end of training
# It keeps the best checkpoint in memory and returns it to the caller.

def train_model(
    model,
    train_loader,
    dev_loader,
    optimizer,
    scheduler,
    loss_fn,
    device,
    num_epochs,
    patience,
    grad_clip,
    run_name="final",
    use_rdrop=False,
    rdrop_alpha=0.5,
    use_swa=False,
    swa_start_epoch=8,
    swa_lr=1e-4,
):
    history = []
    best_state = None
    best_dev_f1 = -1.0
    patience_counter = 0
    start_time = time.time()

    swa_model = AveragedModel(model) if use_swa else None
    swa_scheduler = SWALR(optimizer, swa_lr=swa_lr) if use_swa else None

    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        all_train_probs = []
        all_train_labels = []

        progress = tqdm(train_loader, desc=f"{run_name} epoch {epoch}/{num_epochs}")
        for premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths, labels in progress:
            premise_ids = premise_ids.to(device)
            premise_lengths = premise_lengths.to(device)
            hypothesis_ids = hypothesis_ids.to(device)
            hypothesis_lengths = hypothesis_lengths.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            if use_rdrop:
                logits_1 = model(premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths)
                logits_2 = model(premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths)

                ce_1 = loss_fn(logits_1, labels)
                ce_2 = loss_fn(logits_2, labels)
                kl_loss = bernoulli_symmetric_kl_from_logits(logits_1, logits_2)
                loss = 0.5 * (ce_1 + ce_2) + rdrop_alpha * kl_loss
                logits = 0.5 * (logits_1 + logits_2)
            else:
                logits = model(premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths)
                loss = loss_fn(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            probs = torch.sigmoid(logits)
            running_loss += loss.item()
            all_train_probs.extend(probs.detach().cpu().numpy().tolist())
            all_train_labels.extend(labels.detach().cpu().numpy().tolist())
            progress.set_postfix(loss=f"{loss.item():.4f}")

        if use_swa and epoch >= swa_start_epoch:
            swa_model.update_parameters(model)
            swa_scheduler.step()

        train_metrics, _ = compute_metrics_from_probs(all_train_labels, all_train_probs, threshold=0.5)
        train_metrics["loss"] = float(running_loss / max(1, len(train_loader)))

        eval_model = swa_model.module if (use_swa and epoch >= swa_start_epoch) else model
        dev_metrics_05, dev_labels, dev_probs, _ = evaluate_model(eval_model, dev_loader, loss_fn, device, threshold=0.5)
        best_threshold, best_threshold_metrics, threshold_sweep = find_best_threshold(dev_labels, dev_probs)
        best_threshold_metrics["loss"] = dev_metrics_05["loss"]

        history.append({
            "epoch": epoch,
            "train": train_metrics,
            "dev_threshold_0.5": dev_metrics_05,
            "dev_best_threshold": best_threshold_metrics,
            "best_threshold_value": best_threshold,
        })

        if not (use_swa and epoch >= swa_start_epoch):
            scheduler.step(best_threshold_metrics["f1_macro"])

        print(
            f"Epoch {epoch} | "
            f"train_loss={train_metrics['loss']:.4f} | train_f1={train_metrics['f1_macro']:.4f} | "
            f"dev_f1@best={best_threshold_metrics['f1_macro']:.4f} | "
            f"dev_acc@best={best_threshold_metrics['accuracy']:.4f} | threshold={best_threshold}"
        )

        if best_threshold_metrics["f1_macro"] > best_dev_f1:
            best_dev_f1 = best_threshold_metrics["f1_macro"]
            patience_counter = 0
            state_dict = swa_model.module.state_dict() if (use_swa and epoch >= swa_start_epoch) else model.state_dict()

            best_state = {
                "model_state_dict": {k: v.detach().cpu().clone() for k, v in state_dict.items()},
                "epoch": epoch,
                "best_dev_f1_macro": best_dev_f1,
                "best_threshold": best_threshold,
                "best_dev_metrics": best_threshold_metrics,
                "threshold_sweep": threshold_sweep.to_dict(orient="records"),
                "used_rdrop": use_rdrop,
                "used_swa": use_swa,
            }
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered after epoch {epoch}.")
            break

    total_train_time = time.time() - start_time
    return history, best_state, total_train_time


In [18]:

# Optional shortlist search.
# This is not a large random search; it only compares a small number of hand-
# selected candidate settings and then returns the strongest one.

def run_shortlist_search(train_df, dev_df, vocab, embedding_matrix, actual_embedding_dim):
    """Evaluate a small shortlist of candidate hyperparameter settings."""
    results = []

    for i, cfg in enumerate(SEARCH_SHORTLIST, start=1):
        print(f"=== Shortlist run {i}/{len(SEARCH_SHORTLIST)} ===")
        print(cfg)

        train_loader, dev_loader = build_dataloaders(
            train_df=train_df,
            dev_df=dev_df,
            vocab=vocab,
            max_len=cfg["max_len"],
            batch_size=cfg["batch_size"],
        )

        model = build_model(
            vocab_size=len(vocab),
            embedding_dim=actual_embedding_dim,
            hidden_size=cfg["hidden_size"],
            padding_idx=vocab["<pad>"],
            embedding_matrix=embedding_matrix,
            dropout=cfg["dropout"],
            train_embeddings=cfg["train_embeddings"],
        )

        loss_fn = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

        history, best_state, train_time = train_model(
            model=model,
            train_loader=train_loader,
            dev_loader=dev_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            loss_fn=loss_fn,
            device=device,
            num_epochs=SEARCH_EPOCHS,
            patience=SEARCH_PATIENCE,
            grad_clip=GRAD_CLIP,
            run_name=f"search-{i}",
            use_rdrop=USE_RDROP,
            rdrop_alpha=RDROP_ALPHA,
            use_swa=False,
        )

        row = {
            "trial": i,
            **cfg,
            "best_epoch": best_state["epoch"],
            "best_threshold": best_state["best_threshold"],
            "train_time_seconds": float(train_time),
            "dev_accuracy": float(best_state["best_dev_metrics"]["accuracy"]),
            "dev_f1_macro": float(best_state["best_dev_metrics"]["f1_macro"]),
            "dev_roc_auc": None if best_state["best_dev_metrics"]["roc_auc"] is None else float(best_state["best_dev_metrics"]["roc_auc"]),
        }
        results.append(row)

    results_df = pd.DataFrame(results).sort_values(["dev_f1_macro", "dev_accuracy"], ascending=False).reset_index(drop=True)
    display(results_df)
    return results_df


In [20]:

# Load the training/dev data, inspect dataset statistics, and build the shared
# resources needed by the model bundle.

train_df = pd.read_csv(TRAIN_PATH)
dev_df = pd.read_csv(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("Train label counts:")
print(train_df["label"].value_counts())
print("Dev label counts:")
print(dev_df["label"].value_counts())

train_length_summary = describe_lengths(train_df)
dev_length_summary = describe_lengths(dev_df)
print("Train length summary:", train_length_summary)
print("Dev length summary:", dev_length_summary)

vocab, token_counter = build_vocab(train_df, min_freq=MIN_FREQ, max_size=MAX_VOCAB_SIZE)
print("Vocabulary size:", len(vocab))

embedding_matrix, actual_embedding_dim, embedding_coverage, embedding_source = build_embedding_matrix(vocab)
print("Embedding source:", embedding_source)
print("Embedding dimension:", actual_embedding_dim)
print("Embedding coverage:", round(embedding_coverage, 4))

if RUN_SHORTLIST_SEARCH:
    shortlist_df = run_shortlist_search(train_df, dev_df, vocab, embedding_matrix, actual_embedding_dim)
    best_cfg = shortlist_df.iloc[0].to_dict()
    print("Applying best shortlist configuration:")
    print(best_cfg)
    HIDDEN_SIZE = int(best_cfg["hidden_size"])
    DROPOUT = float(best_cfg["dropout"])
    LEARNING_RATE = float(best_cfg["learning_rate"])
    WEIGHT_DECAY = float(best_cfg["weight_decay"])
    BATCH_SIZE = int(best_cfg["batch_size"])
    MAX_LEN = int(best_cfg["max_len"])
    TRAIN_EMBEDDINGS = bool(best_cfg["train_embeddings"])

train_loader, dev_loader = build_dataloaders(
    train_df=train_df,
    dev_df=dev_df,
    vocab=vocab,
    max_len=MAX_LEN,
    batch_size=BATCH_SIZE,
)

model = build_model(
    vocab_size=len(vocab),
    embedding_dim=actual_embedding_dim,
    hidden_size=HIDDEN_SIZE,
    padding_idx=vocab["<pad>"],
    embedding_matrix=embedding_matrix,
    dropout=DROPOUT,
    train_embeddings=TRAIN_EMBEDDINGS,
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)

history, best_state, total_train_time = train_model(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    loss_fn=loss_fn,
    device=device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    grad_clip=GRAD_CLIP,
    run_name="final",
    use_rdrop=USE_RDROP,
    rdrop_alpha=RDROP_ALPHA,
    use_swa=USE_SWA,
    swa_start_epoch=SWA_START_EPOCH,
    swa_lr=SWA_LR,
)

model_config = {
    "model_id": MODEL_ID,
    "developers": DEVELOPERS,
    "language": LANGUAGE,
    "track_name": TRACK_NAME,
    "notebook_version": NOTEBOOK_VERSION,
    "seed": SEED,
    "lowercase": LOWERCASE,
    "max_len": MAX_LEN,
    "min_freq": MIN_FREQ,
    "max_vocab_size": MAX_VOCAB_SIZE,
    "use_pretrained_embeddings": USE_PRETRAINED_EMBEDDINGS,
    "embedding_backend": EMBEDDING_BACKEND,
    "gensim_model_name": GENSIM_MODEL_NAME,
    "embedding_dim": actual_embedding_dim,
    "train_embeddings": TRAIN_EMBEDDINGS,
    "hidden_size": HIDDEN_SIZE,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs_requested": NUM_EPOCHS,
    "patience": PATIENCE,
    "grad_clip": GRAD_CLIP,
    "device_used": str(device),
    "use_rdrop": USE_RDROP,
    "rdrop_alpha": RDROP_ALPHA,
    "use_swa": USE_SWA,
    "swa_start_epoch": SWA_START_EPOCH,
    "swa_lr": SWA_LR,
    "embedding_source": embedding_source,
    "embedding_coverage": float(embedding_coverage),
}

bundle = {
    "model_state_dict": best_state["model_state_dict"],
    "vocab": vocab,
    "model_config": model_config,
    "best_threshold": float(best_state["best_threshold"]),
    "best_epoch": int(best_state["epoch"]),
    "best_dev_metrics": best_state["best_dev_metrics"],
    "training_history": history,
    "threshold_sweep": best_state["threshold_sweep"],
    "dataset_summary": {
        "train_examples": int(len(train_df)),
        "dev_examples": int(len(dev_df)),
        "train_label_distribution": train_df["label"].value_counts().to_dict(),
        "dev_label_distribution": dev_df["label"].value_counts().to_dict(),
        "train_length_summary": train_length_summary,
        "dev_length_summary": dev_length_summary,
        "vocab_size": int(len(vocab)),
        "total_train_time_seconds": float(total_train_time),
    },
}

torch.save(bundle, MODEL_BUNDLE_PATH)
print(f"Saved model bundle to: {MODEL_BUNDLE_PATH}")
print(f"Best epoch: {bundle['best_epoch']}")
print(f"Best threshold: {bundle['best_threshold']}")
print(f"Best dev macro-F1 during training: {bundle['best_dev_metrics']['f1_macro']:.4f}")


Train shape: (24432, 3)
Dev shape: (6736, 3)
Train label counts:
label
1    12648
0    11784
Name: count, dtype: int64
Dev label counts:
label
1    3478
0    3258
Name: count, dtype: int64
Train length summary: {'count': 48864, 'mean': 17.19079485920105, 'p90': 31, 'p95': 39, 'max': 305}
Dev length summary: {'count': 13472, 'mean': 16.988791567695962, 'p90': 31, 'p95': 39, 'max': 144}
Vocabulary size: 22533
Loading pretrained vectors: glove-wiki-gigaword-100
[==================================================] 100.0% 128.1/128.1MB downloaded
Embedding source: gensim::glove-wiki-gigaword-100
Embedding dimension: 100
Embedding coverage: 0.9627


final epoch 1/12:   0%|          | 0/382 [00:00<?, ?it/s]

Epoch 1 | train_loss=0.6555 | train_f1=0.6050 | dev_f1@best=0.6604 | dev_acc@best=0.6627 | threshold=0.56


final epoch 2/12:   0%|          | 0/382 [00:00<?, ?it/s]

Epoch 2 | train_loss=0.6077 | train_f1=0.6683 | dev_f1@best=0.6822 | dev_acc@best=0.6835 | threshold=0.56


...

Epoch 10 | train_loss=0.4156 | train_f1=0.8370 | dev_f1@best=0.7357 | dev_acc@best=0.7357 | threshold=0.56


final epoch 11/12:   0%|          | 0/382 [00:00<?, ?it/s]

Epoch 11 | train_loss=0.3886 | train_f1=0.8548 | dev_f1@best=0.7355 | dev_acc@best=0.7363 | threshold=0.48


final epoch 12/12:   0%|          | 0/382 [00:00<?, ?it/s]

Epoch 12 | train_loss=0.3634 | train_f1=0.8707 | dev_f1@best=0.7351 | dev_acc@best=0.7360 | threshold=0.48
Saved model bundle to: nli_esim_plus_bundle.pt
Best epoch: 9
Best threshold: 0.54
Best dev macro-F1 during training: 0.7365
